# Zero-shot benchmark of open models (Colab)
Compares open chat models with the GPT-5.4 baseline (64-68% top-1 on the real-style sets) and with our fine-tuned 1.5B run I (about 30%), using the same 186-name menu.

Upload `fallacy_code.zip` and `external_eval.zip` (private, local use only) to the Files sidebar and wait for both to finish. Needs the `HF_TOKEN` secret with *Notebook access* on.

**GPU:** the notebook checks the GPU's memory and loads each model at full precision (bf16) when it fits (L4, A100), or in 4-bit on a 16 GB GPU (T4). Gemma needs bf16, so use an L4 or A100 for it.
**Gated models:** the `unsloth/...` copies of Gemma and Llama need no access request. The official `google/...` and `meta-llama/...` repos do (accept their licence on the model page first).
**Time (rough guess, not measured):** 15-30 minutes per model on a T4 including the download. Run the smoke test first.

In [ ]:
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub bitsandbytes sentencepiece
!pip list 2>/dev/null | grep -iE "^(transformers|torch|bitsandbytes|accelerate|peft) "

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less memory fragmentation
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/external_eval.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!python fetch_data.py --rev v5
!wc -l data/generated/*.jsonl external/*.jsonl

In [ ]:
# Pick the loading mode from the GPU's memory: full precision (bf16) if it fits, 4-bit otherwise.
import torch
gib = torch.cuda.get_device_properties(0).total_memory / 2**30
Q7  = '' if gib >= 20 else '--four-bit'   # 7-9B models need about 15-19 GiB at full precision (fits L4 24 GiB and A100)
Q14 = '' if gib >= 38 else '--four-bit'   # the 14B model needs about 30 GiB at full precision (fits an A100 40 GiB only)
print(f'GPU: {torch.cuda.get_device_name(0)}, {gib:.0f} GiB | 7-9B models: {Q7 or "full precision (bf16)"} | 14B model: {Q14 or "full precision (bf16)"}')

## Smoke test (8 texts): prints timestamped messages, then one result line

In [ ]:
# Smoke test: 8 texts. You should see timestamped messages within seconds, then a result line.
!python -u zeroshot_open.py --model Qwen/Qwen2.5-7B-Instruct {Q7} --limit 8 --sets eval

## Qwen2.5-7B-Instruct

In [ ]:
!python -u zeroshot_open.py --model Qwen/Qwen2.5-7B-Instruct {Q7} 2>&1 | tee zs_qwen7b.log

## Qwen2.5-14B-Instruct (about 2x slower)

In [ ]:
!python -u zeroshot_open.py --model Qwen/Qwen2.5-14B-Instruct {Q14} 2>&1 | tee zs_qwen14b.log

## Gemma-2-9B-it (best on an L4/A100; on a T4 it may give NaNs or garbage, check `unparsed` in the output)

In [ ]:
!python -u zeroshot_open.py --model unsloth/gemma-2-9b-it {Q7} 2>&1 | tee zs_gemma9b.log

## Llama-3.1-8B-Instruct

In [ ]:
!python -u zeroshot_open.py --model unsloth/Llama-3.1-8B-Instruct {Q7} 2>&1 | tee zs_llama8b.log

## Download the results

In [ ]:
!zip -q zeroshot_results.zip data/zeroshot_*.json zs_*.log
from google.colab import files
files.download('/content/work/zeroshot_results.zip')